```
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
```

# RMI Sample Queries: Data Scientist (GA)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Data_Scientist_Samples.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Frmi_sample_queries%2Fnotebooks%2FData_Scientist_Samples.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Data_Scientist_Samples.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Data_Scientist_Samples.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>



This notebook contains sample queries for the **Data Scientist** persona, specifically for the **GA** stage.

## 1. Setup

In [ ]:
from google.colab import auth
import pandas as pd

auth.authenticate_user()

project_id = 'your-project-id' #@param {type:"string"}

### Writable Dataset

Several queries in this notebook (e.g., those creating Materialized Views, Models, or Views) require a **writable dataset** within your own project. 
**Note**: The source `boston_oct_2025_sample_data` dataset is a read-only subscription and cannot be used to store new resources.

Run the cell below to create a new dataset (e.g., `rmi_analysis`) in your project if you haven't already.

**Important**: When running queries that create new BigQuery resources (e.g., tables, views, models) outside of these `%%bigquery` magic cells, remember to manually prepend the job ID with `msqlfactory--` for proper tracking. For example: `bq query --job_id=msqlfactory--your-descriptive-job-name ...`

In [ ]:
dataset_id = "rmi_analysis" #@param {type:"string"}
! bq --location=US mk --dataset {project_id}:{dataset_id}

## GA (General Availability)

### ds1_outlier_detection.sql
**Business Question**: Which travel time records for a specific route are statistical outliers?
Use Case: Automatically flags anomalous data points that could indicate extreme traffic events or potential data collection issues.
Product Stage: GA
Estimated Bytes Processed: ~151 MB (Requires JOIN with routes_status)

In [ ]:
%%bigquery --project {project_id} df_ds1_outlier_detection
/*
  QUALITY FILTERS:
  1. continuous_path: Excludes records where the geometry is not a single ST_LineString.
  2. length_integrity: Excludes records where actual physical length deviates by > 5% 
     from the intended 'route_length' attribute.
*/

WITH quality_filtered_history AS (
  SELECT
    h.selected_route_id,
    h.record_time,
    h.duration_in_seconds,
    ST_LENGTH(h.route_geometry) as actual_length,
    CAST(JSON_VALUE(s.route_attributes, '$.route_length') AS FLOAT64) as intended_length
  FROM `boston_oct_2025_sample_data.historical_travel_time` h
  JOIN `boston_oct_2025_sample_data.routes_status` s USING(selected_route_id)
  WHERE h.selected_route_id = 'route-4202493217'
    AND h.record_time BETWEEN '2025-10-01' AND '2025-11-01'
    -- Quality filter: Only process single, continuous paths
    AND ST_GEOMETRYTYPE(h.route_geometry) = 'ST_LineString'
    -- Quality filter: Length deviation check (< 5%)
    AND SAFE_DIVIDE(ABS(ST_LENGTH(h.route_geometry) - CAST(JSON_VALUE(s.route_attributes, '$.route_length') AS FLOAT64)), CAST(JSON_VALUE(s.route_attributes, '$.route_length') AS FLOAT64)) < 0.05
),
stats AS (
  SELECT
    APPROX_QUANTILES(duration_in_seconds, 100)[OFFSET(25)] AS q1,
    APPROX_QUANTILES(duration_in_seconds, 100)[OFFSET(75)] AS q3
  FROM quality_filtered_history
),
outlier_thresholds AS (
  SELECT
    q1,
    q3,
    (q3 - q1) AS iqr,
    q1 - (1.5 * (q3 - q1)) AS lower_bound,
    q3 + (1.5 * (q3 - q1)) AS upper_bound
  FROM stats
)
SELECT
  h.record_time,
  h.duration_in_seconds,
  t.lower_bound,
  t.upper_bound,
  CASE 
    WHEN h.duration_in_seconds > t.upper_bound THEN 'High_Outlier'
    WHEN h.duration_in_seconds < t.lower_bound THEN 'Low_Outlier'
  END as outlier_type
FROM quality_filtered_history h, outlier_thresholds t
-- Filter for records outside the calculated IQR bounds
WHERE (h.duration_in_seconds > t.upper_bound OR h.duration_in_seconds < t.lower_bound)
ORDER BY h.record_time DESC;


### ds2_similarity_clustering.sql
**Business Question**: Which routes exhibit similar traffic patterns based on their average peak-hour delay ratios?
Use Case: Grouping routes by behavioral similarity allows planners to apply similar mitigation strategies to entire clusters of road segments rather than analyzing each route individually.
Product Stage: GA
Estimated Bytes Processed: ~150 MB

In [ ]:
%%bigquery --project {project_id} df_ds2_similarity_clustering
/*
  INTERPRETATION GUIDE:
  Routes assigned to the same 'cluster_id' share a similar diurnal traffic profile 
  (the relationship between AM, Midday, and PM delays).
  
  Example Interpretation:
  - Cluster 1: Commuter Heavy (High AM/PM delay, low Midday).
  - Cluster 2: Consistently Efficient (Delay ratio near 1.0 all day).
  - Cluster 3: Midday Bottleneck (High Midday delay, typical AM/PM).
*/

-- Step 1: Create the K-Means model.
-- NOTE: The source dataset (e.g., `boston_oct_2025_sample_data`) is a read-only subscription.
-- This model MUST be created in a separate, writable dataset within your project.
-- Replace `your-project.your-dataset` with your target location.

CREATE OR REPLACE MODEL `your-project.your-dataset.route_clusters`
OPTIONS(model_type='kmeans', num_clusters=5) AS
SELECT
  -- K-Means works with numerical features. We will use the delay ratios as features.
  COALESCE(AVG(CASE WHEN EXTRACT(HOUR FROM DATETIME(record_time, 'America/New_York')) BETWEEN 7 AND 9 THEN SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds) END), 1.0) AS avg_am_delay,
  COALESCE(AVG(CASE WHEN EXTRACT(HOUR FROM DATETIME(record_time, 'America/New_York')) BETWEEN 12 AND 14 THEN SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds) END), 1.0) AS avg_midday_delay,
  COALESCE(AVG(CASE WHEN EXTRACT(HOUR FROM DATETIME(record_time, 'America/New_York')) BETWEEN 16 AND 18 THEN SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds) END), 1.0) AS avg_pm_delay
FROM `boston_oct_2025_sample_data.historical_travel_time`
WHERE record_time BETWEEN '2025-10-01' AND '2025-11-01'
GROUP BY selected_route_id;

-- Step 2: Predict the cluster for each route using the trained model.
WITH route_features AS (
  SELECT
    selected_route_id,
    display_name,
    COALESCE(AVG(CASE WHEN EXTRACT(HOUR FROM DATETIME(record_time, 'America/New_York')) BETWEEN 7 AND 9 THEN SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds) END), 1.0) AS avg_am_delay,
    COALESCE(AVG(CASE WHEN EXTRACT(HOUR FROM DATETIME(record_time, 'America/New_York')) BETWEEN 12 AND 14 THEN SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds) END), 1.0) AS avg_midday_delay,
    COALESCE(AVG(CASE WHEN EXTRACT(HOUR FROM DATETIME(record_time, 'America/New_York')) BETWEEN 16 AND 18 THEN SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds) END), 1.0) AS avg_pm_delay
  FROM `boston_oct_2025_sample_data.historical_travel_time`
  WHERE record_time BETWEEN '2025-10-01' AND '2025-11-01'
  GROUP BY 1, 2
)
SELECT
  selected_route_id,
  display_name,
  CENTROID_ID AS cluster_id,
  ROUND(avg_am_delay, 2) as am_ratio,
  ROUND(avg_midday_delay, 2) as midday_ratio,
  ROUND(avg_pm_delay, 2) as pm_ratio
FROM ML.PREDICT(MODEL `your-project.your-dataset.route_clusters`,
  (SELECT * FROM route_features)
)
ORDER BY cluster_id, selected_route_id;


### ds3_feature_engineering.sql
**Business Question**: How can I prepare a high-quality, gap-aware feature set for training a predictive traffic model?
Use Case: Demonstrates how to regularize a time-series using a timestamp grid. This ensures that window functions (LAG, AVG) accurately reflect chronological time even when records are missing due to quality filtering or detours.
Product Stage: GA
Estimated Bytes Processed: ~151 MB

In [ ]:
%%bigquery --project {project_id} df_ds3_feature_engineering
/*
  HANDLING MISSING DATA (DETOURS/GAPS):
  By joining the RMI data with a generated 'time_grid', we identify missing records. 
  Downstream models can then decide how to handle these nulls (e.g., interpolation, 
  imputation, or masking), preventing window functions from 'collapsing' time gaps.
*/

WITH quality_filtered_base AS (
  SELECT
    -- Truncating to hour to match the RMI collection interval
    TIMESTAMP_TRUNC(h.record_time, HOUR) as record_hour,
    h.duration_in_seconds,
    ST_LENGTH(h.route_geometry) as actual_length,
    CAST(JSON_VALUE(s.route_attributes, '$.route_length') AS FLOAT64) as intended_length
  FROM `boston_oct_2025_sample_data.historical_travel_time` h
  JOIN `boston_oct_2025_sample_data.routes_status` s USING(selected_route_id)
  WHERE h.selected_route_id = 'route-4202493217'
    AND h.record_time BETWEEN '2025-10-01' AND '2025-11-01'
    -- Quality filter: Only process single, continuous paths
    AND ST_GEOMETRYTYPE(h.route_geometry) = 'ST_LineString'
),
hourly_averages AS (
  -- Aggregate to a single record per hour before regularizing
  SELECT 
    record_hour,
    AVG(duration_in_seconds) as avg_duration,
    COUNT(*) as samples_in_hour
  FROM quality_filtered_base
  WHERE SAFE_DIVIDE(ABS(actual_length - intended_length), intended_length) < 0.05
  GROUP BY 1
),
time_grid AS (
  -- Generate a continuous hourly grid for the study period
  SELECT hour
  FROM UNNEST(GENERATE_TIMESTAMP_ARRAY('2025-10-01', '2025-10-31', INTERVAL 1 HOUR)) as hour
),
regularized_series AS (
  SELECT
    g.hour,
    a.avg_duration as duration_in_seconds,
    COALESCE(a.samples_in_hour, 0) as samples_in_hour,
    IF(a.avg_duration IS NULL, TRUE, FALSE) as is_missing_data
  FROM time_grid g
  LEFT JOIN hourly_averages a ON g.hour = a.record_hour
)
SELECT
  hour,
  ROUND(duration_in_seconds, 2) as duration_in_seconds,
  samples_in_hour,
  is_missing_data,
  -- Lagged features now accurately represent -1hr and -2hr regardless of data availability
  ROUND(LAG(duration_in_seconds, 1) OVER (ORDER BY hour), 2) AS lag_1hr_duration,
  ROUND(LAG(duration_in_seconds, 2) OVER (ORDER BY hour), 2) AS lag_2hr_duration,
  -- Rolling average (3-hour window)
  ROUND(AVG(duration_in_seconds) OVER (
    ORDER BY hour 
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
  ), 2) AS rolling_avg_3hr
FROM regularized_series
ORDER BY hour DESC;


### ds4_route_integrity_audit.sql
**Business Question**: When and for how long did specific routes experience extreme geometry deviations?
Use Case: Identifies persistent "integrity incidents" rather than transient noise. By grouping consecutive failed records into windows, engineers can correlate failures with specific infrastructure changes, GPS outages, or registration updates.
Product Stage: GA
Estimated Bytes Processed: ~151 MB (Requires JOIN with routes_status)

In [ ]:
%%bigquery --project {project_id} df_ds4_route_integrity_audit
/*
  DEFINITION: Route Integrity
  In RMI, 'Route Integrity' measures the spatial consistency between a route's 
  registered definition and its actual data collection performance.
  
  - The Baseline: 'intended_length' (meters) provided as a custom attribute during registration.
  - The Signal: 'actual_length' (meters) calculated from the captured ST_LineString.
  - High Integrity: A ratio near 1.0 (actual length matches intended definition).
  - Low Integrity: Significant deviations (> 10%) indicate detours, missing road 
    segments, or incorrect metadata registration.
*/

/*
  ANALYTICAL PATTERN: Islands and Gaps
  This query groups consecutive records that exceed a 10% length deviation threshold 
  into discrete failure windows.
*/

WITH base_comparison AS (
  SELECT
    h.selected_route_id,
    h.display_name,
    h.record_time,
    ST_LENGTH(h.route_geometry) AS actual_length,
    CAST(JSON_VALUE(s.route_attributes, '$.route_length') AS FLOAT64) AS intended_length
  FROM `boston_oct_2025_sample_data.historical_travel_time` h
  JOIN `boston_oct_2025_sample_data.routes_status` s USING (selected_route_id)
  WHERE s.status = 'STATUS_RUNNING'
    AND h.record_time BETWEEN '2025-10-01' AND '2025-11-01'
    -- Quality filter: Exclude non-continuous geometries
    AND ST_GEOMETRYTYPE(h.route_geometry) = 'ST_LineString'
),
outlier_flagging AS (
  SELECT
    *,
    -- Flag if deviation exceeds 10% (actual / intended)
    IF(intended_length IS NOT NULL AND (SAFE_DIVIDE(actual_length, intended_length) > 1.1 OR SAFE_DIVIDE(actual_length, intended_length) < 0.9), 1, 0) as is_outlier
  FROM base_comparison
),
streak_identification AS (
  SELECT
    *,
    -- A new streak starts if this is an outlier and the previous record (by time) wasn't
    IF(is_outlier = 1 AND LAG(is_outlier) OVER (PARTITION BY selected_route_id ORDER BY record_time) = 0, 1, 
       IF(is_outlier = 1 AND LAG(is_outlier) OVER (PARTITION BY selected_route_id ORDER BY record_time) IS NULL, 1, 0)) as is_streak_start
  FROM outlier_flagging
),
streak_grouping AS (
  SELECT
    *,
    -- Cumulative sum of starts creates a unique ID for each failure window
    SUM(is_streak_start) OVER (PARTITION BY selected_route_id ORDER BY record_time) as streak_id
  FROM streak_identification
  WHERE is_outlier = 1
)
SELECT
  selected_route_id,
  display_name,
  MIN(record_time) as failure_start,
  MAX(record_time) as failure_end,
  COUNT(*) as consecutive_records,
  -- Average ratio across the window: Severity of the discrepancy
  ROUND(AVG(SAFE_DIVIDE(actual_length, intended_length)), 2) as avg_deviation_ratio,
  -- Identify if the deviation is an over-count (likely detour) or under-count (missing segments)
  IF(AVG(actual_length) > AVG(intended_length), 'OVER_COUNT', 'UNDER_COUNT') as failure_type
FROM streak_grouping
GROUP BY selected_route_id, display_name, streak_id
-- Focus on persistent issues
HAVING consecutive_records >= 1
ORDER BY failure_start DESC, avg_deviation_ratio DESC
LIMIT 50;


### ds5_reliability_ranking.sql
**Business Question**: When and for how long did specific routes experience persistent travel time spikes?
Use Case: Identifies chronic congestion incidents rather than transient variance. By grouping consecutive "slow" records into windows, operators can distinguish between random noise and actionable infrastructure failures or major events.
Product Stage: GA
Estimated Bytes Processed: ~151 MB (Requires JOIN with routes_status)

In [ ]:
%%bigquery --project {project_id} df_ds5_reliability_ranking
/*
  DEFINITION: Route Reliability vs. Route Integrity
  - Route Integrity (DS4): Measures spatial consistency (Actual Geometry vs. Registered Definition).
  - Route Reliability (DS5): Measures temporal performance (Actual Travel Time vs. Free-flow Baseline).
  
  High Reliability means a route's travel time is stable and near its ideal baseline. 
  Low Reliability (this query) indicates persistent periods of 'excess delay' 
  where actual travel times significantly exceed free-flow estimates.
*/

/*
  ANALYTICAL PATTERN: Reliability Gaps (Islands and Gaps)
  1. Calculate a historical baseline per route.
  2. Flag records where travel time exceeds a 'significant delay' threshold (e.g., 1.5x baseline).
  3. Group consecutive flags into discrete failure windows (streaks).
*/

WITH quality_filtered_history AS (
  -- Standard quality filtering to ensure we analyze healthy geometries
  SELECT
    h.selected_route_id,
    h.display_name,
    h.record_time,
    h.duration_in_seconds,
    h.static_duration_in_seconds
  FROM `boston_oct_2025_sample_data.historical_travel_time` h
  JOIN `boston_oct_2025_sample_data.routes_status` s USING(selected_route_id)
  WHERE h.record_time BETWEEN '2025-10-01' AND '2025-11-01'
    AND ST_GEOMETRYTYPE(h.route_geometry) = 'ST_LineString'
    AND SAFE_DIVIDE(ABS(ST_LENGTH(h.route_geometry) - CAST(JSON_VALUE(s.route_attributes, '$.route_length') AS FLOAT64)), CAST(JSON_VALUE(s.route_attributes, '$.route_length') AS FLOAT64)) < 0.05
),
incident_flagging AS (
  SELECT
    *,
    -- Threshold: Travel time is more than 50% above the static (free-flow) baseline
    IF(SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds) > 1.5, 1, 0) as is_incident
  FROM quality_filtered_history
),
streak_identification AS (
  SELECT
    *,
    -- Identify the start of a new consecutive incident window
    IF(is_incident = 1 AND LAG(is_incident) OVER (PARTITION BY selected_route_id ORDER BY record_time) = 0, 1, 
       IF(is_incident = 1 AND LAG(is_incident) OVER (PARTITION BY selected_route_id ORDER BY record_time) IS NULL, 1, 0)) as is_streak_start
  FROM incident_flagging
),
streak_grouping AS (
  SELECT
    *,
    -- Cumulative sum of starts creates a unique ID for each incident window
    SUM(is_streak_start) OVER (PARTITION BY selected_route_id ORDER BY record_time) as streak_id
  FROM streak_identification
  WHERE is_incident = 1
)
SELECT
  selected_route_id,
  display_name,
  MIN(record_time) as incident_start,
  MAX(record_time) as incident_end,
  -- Total consecutive records in this incident window
  COUNT(*) as consecutive_samples,
  -- Average severity of the delay during this window
  ROUND(AVG(SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds)), 2) as avg_delay_ratio
FROM streak_grouping
GROUP BY selected_route_id, display_name, streak_id
-- Focus on persistent unreliability (lasting at least 3 samples)
HAVING consecutive_samples >= 3
ORDER BY avg_delay_ratio DESC, incident_start DESC
LIMIT 50;


### ds6_travel_time_forecasting.sql
**Business Question**: Can we predict next week's peak travel times based on the last 21 days of history?
Use Case: Demonstrates a complete predictive workflow: Training an ARIMA_PLUS model, evaluating its seasonal fit, and performing backtesting against actual results.
Product Stage: GA (Uses BigQuery ML)
Estimated Bytes Processed: ~150 MB

In [ ]:
%%bigquery --project {project_id} df_ds6_travel_time_forecasting
/*
  METHODOLOGY: TIME-SERIES BACKTESTING
  To build trust in a traffic model, we use 'Backtesting'. We split our 31-day 
  sample dataset into two parts:
  1. Training Set (Weeks 1-3): The model 'learns' the route's diurnal and weekly rhythm.
  2. Validation Set (Week 4): We withhold this data from the model, then ask the 
     model to 'forecast' it. Comparing the forecast to reality gives us an 
     empirical accuracy score.
*/

/*
  INTERPRETATION & VISUALIZATION GUIDE:
  
  1. REPORT INTERPRETATION:
     - 'absolute_error': Smaller is better. Measures the magnitude of the prediction 'miss'.
     - 'within_confidence_interval': This is your 'Anomaly Signal'. 
        - 'YES': Traffic is behaving normally/predictably.
        - 'NO': A significant event occurred (accident, weather, gridlock) that 
          exceeded statistical expectations. This is the trigger for operational alerts.
  
  2. RECOMMENDED VISUALIZATIONS:
     - Time-Series Line: Plot 'forecast_seconds' and 'actual_seconds' on the same Y-axis.
     - Confidence Band: Plot 'lower_bound' and 'upper_bound' as a shaded area. Dots 
       (actuals) falling outside this band are your truly actionable traffic incidents.
*/

-- STEP 1: Train the ARIMA_PLUS model using a 3-week window.
-- We use hourly aggregation (AVG) to regularize the input for the ARIMA algorithm.
CREATE OR REPLACE MODEL `your-project.your-dataset.travel_time_forecast_model`
OPTIONS(
  model_type='ARIMA_PLUS',
  time_series_timestamp_col='record_hour',
  time_series_data_col='duration_in_seconds',
  auto_arima=TRUE,          -- Automatically finds the best P, D, Q parameters.
  data_frequency='HOURLY',
  clean_spikes_and_dips=TRUE -- Prevents one-off accidents from skewing the long-term trend.
) AS
SELECT
  TIMESTAMP_TRUNC(record_time, HOUR) as record_hour,
  AVG(duration_in_seconds) as duration_in_seconds
FROM `boston_oct_2025_sample_data.historical_travel_time`
WHERE selected_route_id = 'route-4202493217'
  AND record_time BETWEEN '2025-10-01' AND '2025-10-21'
GROUP BY 1;

-- STEP 2: Evaluate the model's training metrics.
-- This returns AIC, Log Likelihood, and identified seasonal periods (e.g., DAILY).
-- A low AIC relative to other models indicates a better fit.
SELECT * FROM ML.EVALUATE(MODEL `your-project.your-dataset.travel_time_forecast_model`);

-- STEP 3: Compare Forecast vs. Actual for the 4th week (Backtesting).
-- We forecast a 168-hour 'horizon' (7 full days) to match the final week of October.
WITH forecast_data AS (
  SELECT
    forecast_timestamp,
    forecast_value as predicted_duration,
    prediction_interval_lower_bound as lower_bound,
    prediction_interval_upper_bound as upper_bound
  FROM ML.FORECAST(MODEL `your-project.your-dataset.travel_time_forecast_model`,
    STRUCT(168 AS horizon, 0.9 AS confidence_level))
),
actual_data AS (
  -- Aggregate actual withheld data to the same hourly grid for comparison.
  SELECT
    TIMESTAMP_TRUNC(record_time, HOUR) as record_hour,
    AVG(duration_in_seconds) as actual_duration
  FROM `boston_oct_2025_sample_data.historical_travel_time`
  WHERE selected_route_id = 'route-4202493217'
    AND record_time BETWEEN '2025-10-22' AND '2025-10-29'
  GROUP BY 1
)
SELECT
  f.forecast_timestamp,
  ROUND(f.predicted_duration, 1) as forecast_seconds,
  ROUND(a.actual_duration, 1) as actual_seconds,
  -- absolute_error: How many seconds off was the prediction?
  ROUND(ABS(f.predicted_duration - a.actual_duration), 1) as absolute_error,
  -- within_confidence_interval: Was reality within the 90% expected range?
  IF(a.actual_duration BETWEEN f.lower_bound AND f.upper_bound, 'YES', 'NO') as within_confidence_interval,
  -- Include bounds for visualization in tools like Looker Studio or Colab.
  ROUND(f.lower_bound, 1) as lower_bound,
  ROUND(f.upper_bound, 1) as upper_bound
FROM forecast_data f
LEFT JOIN actual_data a ON f.forecast_timestamp = a.record_hour
WHERE a.actual_duration IS NOT NULL
ORDER BY f.forecast_timestamp;


### ds7_zero_shot_forecasting.sql
**Business Question**: Can we immediately forecast next-day traffic for multiple routes without waiting to train individual models?
Use Case: Demonstrates 'Zero-Shot' forecasting using Google's Time Series Foundation Model (TimesFM). Unlike ARIMA, this model uses pre-trained patterns to predict future travel times for an entire cluster of routes simultaneously, even with limited local history.
Product Stage: GA (Uses AI.FORECAST with TimesFM)
Estimated Bytes Processed: ~150 MB

In [ ]:
%%bigquery --project {project_id} df_ds7_zero_shot_forecasting
/*
  ANALYTICAL ADVANTAGE: Foundation Models vs. Traditional Models
  - ARIMA_PLUS (DS6): Requires 'Training' (Learning) on specific route history first.
  - TimesFM (DS7): Uses 'Zero-Shot' inference via AI.FORECAST. It applies global 
    patterns to your data immediately. Ideal for 'Cold Start' (new routes) or 
    scaling to thousands of routes without per-route training overhead.
*/

-- STEP 1: Prepare a 'Context' window of history for multiple routes.
-- Foundation models like TimesFM perform best with 3-7 days of chronological context.
WITH route_context AS (
  SELECT
    selected_route_id,
    TIMESTAMP_TRUNC(record_time, HOUR) as record_hour,
    AVG(duration_in_seconds) as duration_in_seconds
  FROM `boston_oct_2025_sample_data.historical_travel_time`
  -- We pick a 7-day context window for 3 specific routes
  WHERE selected_route_id IN ('route-4202493217', 'route-3850158153', 'route-381361371')
    AND record_time BETWEEN '2025-10-14' AND '2025-10-21'
  GROUP BY 1, 2
)
-- STEP 2: Use AI.FORECAST to generate predictions.
-- Note: TimesFM is a managed foundation model; no CREATE MODEL is required.
SELECT
  *
FROM AI.FORECAST(
  TABLE route_context,
  data_col => 'duration_in_seconds',
  timestamp_col => 'record_hour',
  model => 'TimesFM 2.0',          -- Specify the foundation model version
  id_cols => ['selected_route_id'], -- Forecast each route independently
  horizon => 24,                   -- Forecast 24 hours ahead
  confidence_level => 0.9          -- Generate 90% confidence intervals
)
ORDER BY selected_route_id, forecast_timestamp;
